In [ ]:
# Cài đặt thư viện tldextract để phân tích tên miền (lấy subdomain, domain, suffix)
!pip install tldextract -q

# Cài đặt thư viện python-whois để lấy thông tin (ví dụ: tuổi đời) của tên miền
!pip install python-whois -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 5.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd  # Thư viện xử lý bảng dữ liệu (DataFrame)
import numpy as np   # Thư viện tính toán
import re            # Thư viện Regular Expressions (để tìm mẫu văn bản)
import tldextract    # Thư viện "mổ xẻ" tên miền
import whois         # Thư viện lấy thông tin WHOIS (sẽ dùng ở bước nâng cao)
from urllib.parse import urlparse, parse_qs # Thư viện "mổ xẻ" URL chuẩn
import time          # Dùng để tạm dừng khi gọi API
import os            # Tương tác với hệ điều hành (để quản lý file)

In [ ]:
import pandas as pd

# --- BƯỚC 1: KIỂM TRA ĐỌC FILE ---
try:
    df_raw = pd.read_csv('malicious_phish.csv')
    print(f"Đã đọc file thành công! Có {len(df_raw)} hàng.")
except FileNotFoundError:
    print("LỖI: Không tìm thấy file 'malicious_phish.csv'.")
    print("Hãy đảm bảo bạn đã tải file lên và viết đúng tên file (có .csv).")
    # Dừng ở đây nếu file không có
    raise

# --- BƯỚC 2: KIỂM TRA CỘT 'type' ---
# In ra các giá trị duy nhất trong cột 'type' để xem chúng thực sự là gì
print("\nCác giá trị duy nhất trong cột 'type' (để kiểm tra):")
print(df_raw['type'].unique()[:10]) # In 10 giá trị đầu tiên

# --- BƯỚC 3: LỌC VÀ TẠO NHÃN (BẢN SỬA LỖI) ---

# .str.strip() -> Xóa khoảng trắng thừa ở đầu/cuối
# .str.lower() -> Chuyển tất cả về chữ thường
type_condition = (df_raw['type'].str.strip().str.lower() == 'phishing') | \
                 (df_raw['type'].str.strip().str.lower() == 'benign')

# Dùng .copy() để tạo DataFrame mới, tránh lỗi SettingWithCopyWarning
df_filtered = df_raw[type_condition].copy()

if df_filtered.empty:
    print("\nLỖI: df_filtered bị rỗng. Không tìm thấy 'phishing' hoặc 'benign'.")
    print("Hãy kiểm tra lại các giá trị duy nhất ở trên. Có thể tên nhãn bị sai?")
else:
    # 2. Tạo cột 'label' an toàn bằng .loc
    # Dùng .loc để gán giá trị
    df_filtered.loc[df_filtered['type'].str.strip().str.lower() == 'phishing', 'label'] = 1
    df_filtered.loc[df_filtered['type'].str.strip().str.lower() == 'benign', 'label'] = 0

    # 3. Chọn cột cuối cùng
    df_clean = df_filtered[['url', 'label']]

    print(f"\nDữ liệu gốc có {len(df_raw)} link.")
    print(f"Dữ liệu đã lọc (chỉ phishing/benign) có {len(df_clean)} link.")

    # --- BƯỚC 4: HIỂN THỊ (AN TOÀN) ---
    print("\nXem trước 5 hàng của dữ liệu sạch:")
    # Dùng print() thay vì display() để tránh lỗi gviz
    print(df_clean.head())

Đã đọc file thành công! Có 408507 hàng.

Các giá trị duy nhất trong cột 'type' (để kiểm tra):
['phishing' 'benign' 'defacement' 'malware' nan]

Dữ liệu gốc có 408507 link.
Dữ liệu đã lọc (chỉ phishing/benign) có 322961 link.

Xem trước 5 hàng của dữ liệu sạch:
                                                 url  label
0                                   br-icloud.com.br    1.0
1                mp3raid.com/music/krizz_kaliko.html    0.0
2                    bopsecrets.org/rexroth/cr/1.htm    0.0
5  http://buzzfil.net/m/show-art/ils-etaient-loin...    0.0
6      espn.go.com/nba/player/_/id/3457/brandon-rush    0.0


In [ ]:
# Đặc trưng dựa trên Độ dài (Length Features)
from urllib.parse import urlparse

print("Bắt đầu trích xuất đặc trưng độ dài...")

# 1. Độ dài toàn bộ URL
df_clean['feat_url_length'] = df_clean['url'].apply(len)

# 2. Độ dài của Tên miền (hostname)
df_clean['feat_hostname_length'] = df_clean['url'].apply(lambda x: len(urlparse(x).netloc))

# 3. Độ dài của Đường dẫn (path)
df_clean['feat_path_length'] = df_clean['url'].apply(lambda x: len(urlparse(x).path))

print("Đã trích xuất xong 3 đặc trưng độ dài!")

# Kiểm tra kết quả (xem 3 cột mới ở cuối)
print(df_clean.head())

Bắt đầu trích xuất đặc trưng độ dài...
Đã trích xuất xong 3 đặc trưng độ dài!
                                                 url  label  feat_url_length  \
0                                   br-icloud.com.br    1.0               16   
1                mp3raid.com/music/krizz_kaliko.html    0.0               35   
2                    bopsecrets.org/rexroth/cr/1.htm    0.0               31   
5  http://buzzfil.net/m/show-art/ils-etaient-loin...    0.0              118   
6      espn.go.com/nba/player/_/id/3457/brandon-rush    0.0               45   

   feat_hostname_length  feat_path_length  
0                     0                16  
1                     0                35  
2                     0                31  
5                    11               100  
6                     0                45  


In [ ]:
# Đặc trưng dựa trên Ký tự Đặc biệt (Special Char Features)
df_clean['feat_count_hyphen'] = df_clean['url'].apply(lambda x: x.count('-'))
df_clean['feat_count_at'] = df_clean['url'].apply(lambda x: x.count('@'))
df_clean['feat_count_dot'] = df_clean['url'].apply(lambda x: x.count('.'))
df_clean['feat_count_slash'] = df_clean['url'].apply(lambda x: x.count('/'))
df_clean['feat_count_percent'] = df_clean['url'].apply(lambda x: x.count('%'))
df_clean['feat_count_digits'] = df_clean['url'].apply(lambda x: sum(c.isdigit() for c in x))
df_clean['feat_count_letters'] = df_clean['url'].apply(lambda x: sum(c.isalpha() for c in x))


In [ ]:
SENSITIVE_KEYWORDS = ['login', 'secure', 'bank', 'account', 'verify', 'password', 'signin']


# Tạo một cột cho mỗi từ khóa (1 nếu có, 0 nếu không)
for keyword in SENSITIVE_KEYWORDS:
    df_clean[f'feat_has_{keyword}'] = df_clean['url'].apply(lambda x: 1 if keyword in x.lower() else 0)


# Hoặc tạo một cột tổng số từ khóa nhạy cảm
df_clean['feat_total_keywords'] = df_clean['url'].apply(lambda x: sum(1 for kw in SENSITIVE_KEYWORDS if kw in x.lower()))


In [ ]:
# Dùng thư viện tldextract        Đặc trưng Tên miền (Domain Features)
def get_domain_features(url):
    try:
        # Tách URL: 'http://login.mail.google.com.phishing.net/page'
        tld = tldextract.extract(url)
        # tld.subdomain = 'login.mail.google.com'
        # tld.domain = 'phishing'
        # tld.suffix = 'net'

        # Đếm số lượng tên miền con
        subdomain_count = len(tld.subdomain.split('.')) if tld.subdomain else 0

        return pd.Series({
            'feat_subdomain_count': subdomain_count,
            'feat_domain_length': len(tld.domain),
            'feat_suffix_length': len(tld.suffix)
        })
    except Exception as e:
        # Trả về 0 nếu URL bị lỗi
        return pd.Series({
            'feat_subdomain_count': 0,
            'feat_domain_length': 0,
            'feat_suffix_length': 0
        })


# Áp dụng hàm này (có thể hơi chậm một chút)
domain_features_df = df_clean['url'].apply(get_domain_features)


# Nối các cột đặc trưng tên miền mới vào bảng chính
df_clean = pd.concat([df_clean, domain_features_df], axis=1)


# Kiểm tra xem có dùng HTTPS không
df_clean['feat_use_https'] = df_clean['url'].apply(lambda x: 1 if urlparse(x).scheme == 'https' else 0)


# Kiểm tra xem có dùng IP thay tên miền không
df_clean['feat_use_ip'] = df_clean['url'].apply(lambda x: 1 if re.match(r".*\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}.*", urlparse(x).netloc) else 0)


In [ ]:
# --- BƯỚC 4: HOÀN THIỆN & BÀN GIAO SẢN PHẨM ---
print("Bắt đầu hoàn thiện bộ dữ liệu...")

# 1. Lọc ra các cột đặc trưng (features) và cột nhãn (label)
# Model ML không thể đọc cột 'url' (dạng chữ)
# Chúng ta cần chọn tất cả các cột bắt đầu bằng 'feat_' và cột 'label'.

# Lấy danh sách tất cả các cột hiện có trong df_clean
all_columns = df_clean.columns.tolist()

# Lọc ra danh sách các cột đặc trưng
feature_columns = [col for col in all_columns if col.startswith('feat_')]

# Cột nhãn
label_column = ['label']

# Kết hợp lại thành danh sách các cột cuối cùng cho model
final_columns_for_model = feature_columns + label_column

# 2. Tạo DataFrame cuối cùng
df_final_dataset = df_clean[final_columns_for_model]

# 3. Dọn dẹp cuối cùng (Rất quan trọng)
# Xóa bất kỳ hàng nào có thể bị lỗi (NaN) trong quá trình trích xuất
# (Model ML sẽ bị crash nếu có giá trị NaN)
df_final_dataset = df_final_dataset.dropna()

# 4. Lưu file CSV
OUTPUT_FILENAME = "dataset_features_final.csv"
df_final_dataset.to_csv(OUTPUT_FILENAME, index=False)

print(f"\n--- HOÀN THÀNH (5A) ---")
print(f"Đã lưu bộ dữ liệu huấn luyện (training dataset) vào file: {OUTPUT_FILENAME}")
print(f"Kích thước cuối cùng: {df_final_dataset.shape[0]} hàng x {df_final_dataset.shape[1]} cột")

# In 5 hàng đầu của file cuối cùng để kiểm tra
print("\nXem trước 5 hàng của dữ liệu cuối cùng:")
print(df_final_dataset.head())

Bắt đầu hoàn thiện bộ dữ liệu...

--- HOÀN THÀNH (5A) ---
Đã lưu bộ dữ liệu huấn luyện (training dataset) vào file: dataset_features_final.csv
Kích thước cuối cùng: 322961 hàng x 24 cột

Xem trước 5 hàng của dữ liệu cuối cùng:
   feat_url_length  feat_hostname_length  feat_path_length  feat_count_hyphen  \
0               16                     0                16                  1   
1               35                     0                35                  0   
2               31                     0                31                  0   
5              118                    11               100                 16   
6               45                     0                45                  1   

   feat_count_at  feat_count_dot  feat_count_slash  feat_count_percent  \
0              0               2                 0                   0   
1              0               2                 2                   0   
2              0               2                 3              

In [ ]:
import pandas as pd

# Đọc LẠI file .csv mà bạn vừa lưu ở Bước 4
try:
    df_check = pd.read_csv("dataset_features_final.csv")

    print("--- KIỂM TRA FILE: dataset_features_final.csv ---")

    # 1. In ra danh sách tất cả các cột (features)
    print("\nDanh sách các cột (features) trong file:")
    print(df_check.columns.tolist())

    # 2. In ra kích thước
    print(f"\nKích thước file (Hàng x Cột): {df_check.shape}")

except FileNotFoundError:
    print("LỖI: Không tìm thấy file 'dataset_features_final.csv'.")
    print("Bạn hãy chạy lại ô code 'BƯỚC 4' trước nhé.")

--- KIỂM TRA FILE: dataset_features_final.csv ---

Danh sách các cột (features) trong file:
['feat_url_length', 'feat_hostname_length', 'feat_path_length', 'feat_count_hyphen', 'feat_count_at', 'feat_count_dot', 'feat_count_slash', 'feat_count_percent', 'feat_count_digits', 'feat_count_letters', 'feat_has_login', 'feat_has_secure', 'feat_has_bank', 'feat_has_account', 'feat_has_verify', 'feat_has_password', 'feat_has_signin', 'feat_total_keywords', 'feat_subdomain_count', 'feat_domain_length', 'feat_suffix_length', 'feat_use_https', 'feat_use_ip', 'label']

Kích thước file (Hàng x Cột): (322961, 24)


5A - Data Dictionary (Mô tả Đặc trưng)
Đây là tài liệu mô tả chi tiết các đặc trưng (features) đã được trích xuất và lưu trong file dataset_features_final.csv.

Nhóm 1: Đặc trưng dựa trên Độ dài
feat_url_length: Tổng độ dài của toàn bộ chuỗi URL.

feat_hostname_length: Độ dài của tên miền (ví dụ: www.google.com).

feat_path_length: Độ dài của đường dẫn (ví dụ: /search/page.html).

Nhóm 2: Đặc trưng dựa trên Ký tự
feat_count_hyphen: Số lượng dấu gạch ngang (-) trong URL.

feat_count_at: Số lượng ký tự (@) trong URL.

feat_count_dot: Số lượng dấu chấm (.) trong URL.

feat_count_slash: Số lượng dấu gạch chéo (/) trong URL.

feat_count_percent: Số lượng ký tự phần trăm (%) (thường dùng trong mã hóa URL).

feat_count_digits: Tổng số lượng chữ số (0-9) trong URL.

feat_count_letters: Tổng số lượng chữ cái (a-z) trong URL.

Nhóm 3: Đặc trưng dựa trên Từ khóa
feat_has_login: Bằng 1 nếu URL chứa từ 'login', 0 nếu không.

feat_has_secure: Bằng 1 nếu URL chứa từ 'secure', 0 nếu không.

feat_has_bank: Bằng 1 nếu URL chứa từ 'bank', 0 nếu không.

feat_has_account: Bằng 1 nếu URL chứa từ 'account', 0 nếu không.

feat_has_verify: Bằng 1 nếu URL chứa từ 'verify', 0 nếu không.

feat_has_password: Bằng 1 nếu URL chứa từ 'password', 0 nếu không.

feat_has_signin: Bằng 1 nếu URL chứa từ 'signin', 0 nếu không.

feat_total_keywords: Tổng số lần các từ khóa nhạy cảm (liệt kê ở trên) xuất hiện trong URL.

Nhóm 4: Đặc trưng dựa trên Tên miền
feat_subdomain_count: Số lượng tên miền con. (Ví dụ: mail.google.com có 2 tên miền con là mail và google).

feat_domain_length: Độ dài của tên miền chính (ví dụ: google trong google.com).

feat_suffix_length: Độ dài của hậu tố tên miền (ví dụ: com trong google.com).

feat_use_https: Bằng 1 nếu URL dùng giao thức https (an toàn), 0 nếu không (ví dụ: dùng http).

feat_use_ip: Bằng 1 nếu tên miền là một địa chỉ IP (ví dụ: http://192.168.1.1/login), 0 nếu không.

Nhãn (Label) - Cột Mục tiêu
label: Cột nhãn mà model cần dự đoán.

1 = Phishing (Lừa đảo)

0 = Benign (An toàn)